# PCA Dynamic NWB-Only (GPU)

NWB-only PCA pipeline with optional GPU acceleration.

- Data loading + spike binning: CPU (robust for irregular spike times)
- PCA computation: GPU when available (`cuML`), otherwise CPU fallback


## GPU Notes

GPU helps most in PCA steps. It does **not** remove RAM/VRAM limits caused by extremely large `trials` tensors.
Keep memory controls (`MAX_EVENTS`, `MAX_TENSOR_GB`) enabled.


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pynwb import NWBHDF5IO
from scipy.ndimage import gaussian_filter1d
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA as SKPCA

sns.set_context("talk")
sns.set_style("white")
warnings.filterwarnings("ignore")


In [ ]:
# -----------------------------
# USER CONFIG
# -----------------------------
NWB_PATH = Path(r"G:\Grant\neuropixels\nwb\Reach15")

WINDOW_START_S = -0.5
WINDOW_END_S = 3.5
BIN_SIZE_S = 0.05
N_COMPONENTS = 12
SMOOTH_SIGMA = 2

LABEL_MODE = "block"   # 'block' or 'stimulus'
EVENT_INCLUDE = None    # e.g. ['reach']

# Memory controls
MAX_EVENTS = 5000
EVENT_SUBSAMPLE_MODE = "uniform"  # 'uniform' or 'first'
MAX_TENSOR_GB = 8.0

# Unit filters
PROBE_FILTER = None
KSLABEL_FILTER = None
UNIT_FILTER_QUERY = None
MAX_UNITS = None

# GPU controls
USE_GPU = True
GPU_PCA_BACKEND = "cuml"   # currently: 'cuml'

OUT_DIR = Path.cwd() / "extra_files"
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Optional GPU backend detection
gpu_ready = False
gpu_backend = None

if USE_GPU and GPU_PCA_BACKEND == "cuml":
    try:
        import cupy as cp
        from cuml.decomposition import PCA as CuPCA
        gpu_ready = True
        gpu_backend = "cuml"
    except Exception as e:
        print("GPU backend unavailable; falling back to CPU:", e)

print("GPU ready:", gpu_ready, "backend:", gpu_backend)


In [ ]:
class nwb_loader:
    def __init__(self, nwb_path):
        self.nwb_path = str(nwb_path)
        self.io = None
        self.nwb = None
        self.load_nwb()

    def load_nwb(self):
        self.io = NWBHDF5IO(self.nwb_path, "r", load_namespaces=True)
        self.nwb = self.io.read()
        return self.nwb

    def trials(self):
        return self.nwb.trials.to_dataframe() if self.nwb.trials is not None else pd.DataFrame()

    def units(self):
        return self.nwb.units.to_dataframe() if self.nwb.units is not None else pd.DataFrame()

    def close(self):
        try:
            if self.io is not None:
                self.io.close()
        except Exception:
            pass


In [ ]:
def resolve_nwb_path(p: Path) -> Path:
    if not p.exists():
        raise FileNotFoundError(f"NWB path not found: {p}")
    if p.is_file():
        return p
    nwb_files = sorted(list(p.rglob("*.nwb")))
    if len(nwb_files) == 1:
        return nwb_files[0]
    if len(nwb_files) > 1:
        raise ValueError("Multiple NWB files found. Point NWB_PATH to one file.")
    files = [x for x in p.iterdir() if x.is_file()]
    if len(files) == 1:
        return files[0]
    raise ValueError("Could not resolve a unique NWB file path.")


def choose_event_time_col(df: pd.DataFrame) -> str:
    for c in ["start_time", "time", "event_time", "timestamps"]:
        if c in df.columns:
            return c
    raise ValueError(f"No event-time column found. Columns: {list(df.columns)}")


def normalize_kslabel(v):
    if pd.isna(v):
        return np.nan
    try:
        iv = int(float(v))
        if iv == 2:
            return "good"
        if iv == 1:
            return "mua"
    except Exception:
        pass
    s = str(v).strip().lower()
    if s in {"2", "good", "single", "singleunit", "single_unit"}:
        return "good"
    if s in {"1", "mua", "multi", "multiunit", "multi_unit"}:
        return "mua"
    return s


def infer_is_opto(df_trials: pd.DataFrame) -> pd.Series:
    if "optogenetics_LED_state" in df_trials.columns:
        c = df_trials["optogenetics_LED_state"]
        if pd.api.types.is_numeric_dtype(c):
            return pd.to_numeric(c, errors="coerce").fillna(0) > 0
        return c.astype(str).str.lower().isin(["1", "true", "on", "high"])
    if "stimulus" in df_trials.columns:
        s = df_trials["stimulus"].astype(str).str.lower()
        return s.str.contains("opto|laser|led|stim", regex=True)
    return pd.Series([False] * len(df_trials), index=df_trials.index)


def build_block_labels(is_opto: pd.Series) -> pd.DataFrame:
    is_opto = is_opto.astype(bool).reset_index(drop=True)
    block_id = (is_opto != is_opto.shift(1, fill_value=is_opto.iloc[0])).cumsum()
    mapping, seen_opto, ok, wk = {}, False, 1, 1
    for b in block_id.unique():
        state = bool(is_opto[block_id == b].iloc[0])
        if state:
            mapping[b] = f"opto_epoch_{ok}"; ok += 1; seen_opto = True
        else:
            mapping[b] = "baseline" if not seen_opto else f"washout_epoch_{wk}"
            if seen_opto: wk += 1
    return pd.DataFrame({"block_id": block_id, "is_opto": is_opto, "block_label": block_id.map(mapping)})


def find_probe_col(df: pd.DataFrame):
    for c in ["probe", "probe_name", "probe_id", "probe_letter", "electrode_group"]:
        if c in df.columns:
            return c
    return None


def find_kslabel_col(df: pd.DataFrame):
    for c in ["KSLabel", "kslabel", "ks_label", "label", "quality"]:
        if c in df.columns:
            return c
    return None


def pick_region_col(df: pd.DataFrame):
    for c in ["brain_region", "location", "region", "acronym", "structure", "ccf_acronym"]:
        if c in df.columns:
            return c
    return None


In [ ]:
resolved_nwb = resolve_nwb_path(NWB_PATH)
mouse = nwb_loader(resolved_nwb)
df_trials = mouse.trials().reset_index(drop=True)
df_units = mouse.units().reset_index(drop=True)

print("NWB:", resolved_nwb)
print("trials:", df_trials.shape, "units:", df_units.shape)


In [ ]:
if df_trials.empty:
    raise ValueError("NWB trials table is empty.")

stim_df = df_trials.copy()
time_col = choose_event_time_col(stim_df)
stim_df["event_time_s"] = pd.to_numeric(stim_df[time_col], errors="coerce")
stim_df = stim_df.dropna(subset=["event_time_s"]).sort_values("event_time_s").reset_index(drop=True)

if EVENT_INCLUDE is not None and "stimulus" in stim_df.columns:
    s = stim_df["stimulus"].astype(str).str.lower()
    m = pd.Series(False, index=stim_df.index)
    for token in EVENT_INCLUDE:
        m = m | s.str.contains(str(token).lower(), regex=False)
    stim_df = stim_df[m].reset_index(drop=True)

n_before = len(stim_df)
if MAX_EVENTS is not None and n_before > MAX_EVENTS:
    if EVENT_SUBSAMPLE_MODE == "first":
        keep = np.arange(MAX_EVENTS)
    else:
        keep = np.linspace(0, n_before - 1, MAX_EVENTS, dtype=int)
    stim_df = stim_df.iloc[keep].reset_index(drop=True)
    print(f"Downsampled events: {n_before} -> {len(stim_df)} (mode={EVENT_SUBSAMPLE_MODE})")

blk = build_block_labels(infer_is_opto(stim_df))
stim_df = pd.concat([stim_df, blk], axis=1)
stim_df["trial_index"] = np.arange(len(stim_df), dtype=int)
stim_df["label"] = stim_df["stimulus"].astype(str) if (LABEL_MODE == "stimulus" and "stimulus" in stim_df.columns) else stim_df["block_label"].astype(str)

print(stim_df[["trial_index", "event_time_s", "label"]].head())
print("events used:", len(stim_df))


In [ ]:
if df_units.empty:
    raise ValueError("NWB units table is empty.")
if "spike_times" not in df_units.columns:
    raise ValueError("NWB units table missing spike_times.")

unit_meta = df_units.copy().reset_index(drop=True)
if "unit_id" not in unit_meta.columns:
    unit_meta["unit_id"] = unit_meta.index.astype(int)

probe_col = find_probe_col(unit_meta)
if PROBE_FILTER is not None:
    if probe_col is None:
        raise ValueError("PROBE_FILTER set but no probe column found.")
    pnorm = unit_meta[probe_col].astype(str).str.strip().str.upper().str[0]
    unit_meta = unit_meta[pnorm == str(PROBE_FILTER).strip().upper()].reset_index(drop=True)

ks_col = find_kslabel_col(unit_meta)
if KSLABEL_FILTER is not None:
    if ks_col is None:
        raise ValueError("KSLABEL_FILTER set but no KS label column found.")
    target = normalize_kslabel(KSLABEL_FILTER)
    unit_meta = unit_meta[unit_meta[ks_col].map(normalize_kslabel) == target].reset_index(drop=True)

if UNIT_FILTER_QUERY:
    unit_meta = unit_meta.query(UNIT_FILTER_QUERY).reset_index(drop=True)
if MAX_UNITS is not None:
    unit_meta = unit_meta.iloc[:MAX_UNITS].reset_index(drop=True)

spike_times_by_unit = [np.asarray(st, dtype=float) for st in unit_meta["spike_times"].values]
if len(spike_times_by_unit) == 0:
    raise ValueError("No units left after filtering.")

print("units used:", len(spike_times_by_unit))


In [ ]:
def bin_spikes_around_events(spike_times_list, event_times_s, win_start_s, win_end_s, bin_size_s, max_tensor_gb=8.0):
    edges = np.arange(win_start_s, win_end_s + bin_size_s, bin_size_s)
    n_bins = len(edges) - 1
    n_trials = len(event_times_s)
    n_units = len(spike_times_list)

    est_gb = (n_trials * n_units * n_bins * np.dtype(np.float32).itemsize) / (1024**3)
    print(f"Requested tensor shape=({n_trials}, {n_units}, {n_bins}) est_mem={est_gb:.2f} GB")
    if est_gb > max_tensor_gb:
        raise MemoryError(
            f"Estimated tensor memory {est_gb:.2f} GB exceeds MAX_TENSOR_GB={max_tensor_gb}. "
            "Reduce events/units or increase BIN_SIZE_S."
        )

    X = np.zeros((n_trials, n_units, n_bins), dtype=np.float32)
    for u, st in enumerate(spike_times_list):
        if st.size == 0:
            continue
        for t, t0 in enumerate(event_times_s):
            i0 = np.searchsorted(st, t0 + win_start_s, side="left")
            i1 = np.searchsorted(st, t0 + win_end_s, side="right")
            rel = st[i0:i1] - t0
            if rel.size:
                counts, _ = np.histogram(rel, bins=edges)
                X[t, u, :] = counts / bin_size_s
    return X, edges[:-1]

trials, time = bin_spikes_around_events(
    spike_times_by_unit,
    stim_df["event_time_s"].to_numpy(dtype=float),
    WINDOW_START_S,
    WINDOW_END_S,
    BIN_SIZE_S,
    max_tensor_gb=MAX_TENSOR_GB,
)
print("trials shape:", trials.shape)


In [ ]:
def run_pca(X, n_components=12, use_gpu=False):
    # X shape expected: (features, samples)
    if use_gpu and gpu_ready and gpu_backend == "cuml":
        Xg = cp.asarray(X.T, dtype=cp.float32)  # (samples, features)
        Xg = (Xg - Xg.mean(axis=0)) / (Xg.std(axis=0) + 1e-8)
        pca = CuPCA(n_components=min(n_components, Xg.shape[0], Xg.shape[1]))
        scores = pca.fit_transform(Xg)  # (samples, comps)
        Xp = cp.asnumpy(scores).T
        evr = cp.asnumpy(pca.explained_variance_ratio_)
        return Xp, evr, "gpu-cuml"

    Xz = StandardScaler(with_mean=True, with_std=True).fit_transform(X.T).T
    pca = SKPCA(n_components=min(n_components, Xz.shape[0], Xz.shape[1]))
    Xp = pca.fit_transform(Xz.T).T
    return Xp, pca.explained_variance_ratio_, "cpu-sklearn"

trial_type = stim_df["label"].to_numpy()
trial_types = pd.unique(trial_type)
t_type_ind = [np.where(trial_type == t)[0] for t in trial_types]

in_task = (time >= 0.0) & (time < (WINDOW_END_S - BIN_SIZE_S))
X_trial = np.vstack([trials[i, :, in_task].mean(axis=1) for i in range(trials.shape[0])]).T
Xp, evr_trial, backend_used = run_pca(X_trial, n_components=N_COMPONENTS, use_gpu=USE_GPU)

print("PCA backend:", backend_used)
print("Trial PCA EVR first 5:", np.round(evr_trial[:5], 4))


In [ ]:
projections = [(0, 1), (1, 2), (0, 2)]
pal = sns.color_palette("colorblind", len(trial_types))
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (i, j) in zip(axes, projections):
    for k, t in enumerate(trial_types):
        idx = t_type_ind[k]
        ax.scatter(Xp[i, idx], Xp[j, idx], s=28, alpha=0.8, color=pal[k], label=str(t))
    ax.set_xlabel(f"PC {i+1}")
    ax.set_ylabel(f"PC {j+1}")
axes[-1].legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
trial_averages = []
kept_labels = []
for t, idx in zip(trial_types, t_type_ind):
    if len(idx) > 0:
        trial_averages.append(trials[idx].mean(axis=0))
        kept_labels.append(t)

if len(trial_averages) < 2:
    raise ValueError("Need >=2 labels for trajectory PCA.")

Xa = np.hstack(trial_averages)
Xa_p, evr_traj, backend_traj = run_pca(Xa, n_components=N_COMPONENTS, use_gpu=USE_GPU)
print("Trajectory PCA backend:", backend_traj)
print("Trajectory EVR first 5:", np.round(evr_traj[:5], 4))

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True)
n_bins = trials.shape[2]
for comp in range(min(3, Xa_p.shape[0])):
    ax = axes[comp]
    for k, lbl in enumerate(kept_labels):
        s = k * n_bins
        e = (k + 1) * n_bins
        x = Xa_p[comp, s:e]
        if SMOOTH_SIGMA and SMOOTH_SIGMA > 0:
            x = gaussian_filter1d(x, sigma=SMOOTH_SIGMA)
        ax.plot(time, x, lw=2, color=pal[k], label=str(lbl))
    ax.axvline(0, color="gray", ls="--", lw=1)
    ax.set_ylabel(f"PC {comp+1}")

axes[1].set_xlabel("Time from event (s)")
axes[-1].legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
region_col = pick_region_col(unit_meta)
if region_col is None:
    print("No region column found; skipping region-level PCA.")
else:
    rs = unit_meta[region_col].astype(str).fillna("unknown")
    keep_regions = rs.value_counts()
    keep_regions = keep_regions[keep_regions >= 3].index

    if len(keep_regions) < 2:
        print("Not enough regions with >=3 units; skipping.")
    else:
        region_ids = [np.where(rs.values == r)[0] for r in keep_regions]
        region_trials = np.stack([trials[:, ridx, :].mean(axis=1) for ridx in region_ids], axis=1)
        X_region = np.vstack([region_trials[i, :, in_task].mean(axis=1) for i in range(region_trials.shape[0])]).T
        Xrp, evr_region, backend_region = run_pca(X_region, n_components=6, use_gpu=USE_GPU)
        print("Region PCA backend:", backend_region)
        print("Region EVR first 5:", np.round(evr_region[:5], 4))

        plt.figure(figsize=(5, 4))
        for k, t in enumerate(trial_types):
            idx = t_type_ind[k]
            plt.scatter(Xrp[0, idx], Xrp[1, idx], s=28, alpha=0.8, color=pal[k], label=str(t))
        plt.xlabel("PC 1")
        plt.ylabel("PC 2")
        plt.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
        sns.despine()
        plt.tight_layout()
        plt.show()


In [ ]:
np.save(OUT_DIR / "nwb_only_gpu_trials_fr.npy", trials)
np.save(OUT_DIR / "nwb_only_gpu_trial_pca_scores.npy", Xp)
np.save(OUT_DIR / "nwb_only_gpu_traj_pca_scores.npy", Xa_p)
stim_df.to_csv(OUT_DIR / "nwb_only_gpu_trial_table.csv", index=False)
unit_meta.to_csv(OUT_DIR / "nwb_only_gpu_unit_meta.csv", index=False)

print("Saved files to:", OUT_DIR)
mouse.close()
